# Reading from Bronze


### Init


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, size, split
from pyspark.sql.functions import year, month, dayofmonth
from pyspark.sql.functions import to_date
from pyspark.sql.functions import col, sum as spark_sum
from pyspark.sql.functions import to_date, coalesce, col

In [0]:
def log(step, message):
    print(f"[{step}] {message}")

### loading Bronze 


In [0]:
log("Silver", "Loading bronze data")

df = spark.table("workspace.bronze.sales_details")


# Transformations


## Schema Validation

In [0]:
required_cols = ["Row_ID","Order_ID","Order_Date","Ship_Date","Ship_Mode","Customer_ID","Customer_Name","Segment","Country","City","execution_datetime","source_file"]

for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

df = df.select(*required_cols)

log("Silver", "Schema validation passed")

### Renaming Columns

In [0]:
log("Silver", "Applying column rename map")

RENAME_MAP = {
    "Row_ID": "row_id",
    "Order_ID": "order_id",
    "Order_Date": "order_date",
    "Ship_Date": "ship_date",
    "Ship_Mode": "ship_mode",
    "Customer_ID": "customer_id",
    "Customer_Name": "customer_name",
    "Segment": "segment",
    "Country": "country",
    "City": "city",
    "State": "state"
}

df = df.select([
    col(c).alias(RENAME_MAP.get(c, c.lower()))
    for c in df.columns
])

log("Silver", f"Columns after rename: {df.columns}")

## Data Normalization

### Removing Duplicates

In [0]:
log("Silver", "Checking duplicates based on order_id")

#find duplicate order_ids
duplicates = (
    df.groupBy("order_id")
      .count()
      .filter(col("count") > 1)
)

dup_count = duplicates.count()
log("Silver", f"Number of duplicate order_ids: {dup_count}")

#show duplicate rows
if dup_count > 0:
    log("Silver", "Displaying duplicate rows")

    duplicate_rows = df.join(duplicates, on="order_id", how="inner")
    duplicate_rows.display()

In [0]:
log("Silver", "Removing duplicates based on order_id")

before_dedup = df.count()
df = df.dropDuplicates(["order_id"])
after_dedup = df.count()

log("Silver", f"Rows after dedup: {after_dedup} (removed {before_dedup - after_dedup})")

## Data Cleaning/Formatting


### Handling missing data

In [0]:
#calculating number of missing values in each column by converting into 1s and summing them up
log("Silver", "Checking for missing values")

null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.display()

### Standardizing data formats

In [0]:
log("Silver", "Converting date columns")

def parse_date(column):
    return coalesce(
        to_date(col(column), "MM/dd/yyyy"),
        to_date(col(column), "yyyy-MM-dd"),
        to_date(col(column), "dd-MM-yyyy"),
        to_date(col(column), "MM-dd-yy")
    )

df = df.withColumn("order_date", parse_date("order_date"))
df = df.withColumn("ship_date", parse_date("ship_date"))

In [0]:
# removing NULL order_date and ship_date and ensuring ship_date is not before order_date
log("Silver", "Checking for invalid shipment dates")

before = df.count()

invalid_rows = df.filter(
    (col("order_date").isNull()) |
    (col("ship_date").isNull()) |
    (col("ship_date") < col("order_date"))
).count()

log("Silver", f"Invalid date rows detected: {invalid_rows}")

df = df.filter(
    (col("order_date").isNotNull()) &
    (col("ship_date").isNotNull()) &
    (col("ship_date") >= col("order_date"))
)

after = df.count()

log("Silver", f"Rows after date cleaning: {after} (removed {before - after})")

In [0]:
cols_to_check = ["segment", "ship_mode", "country", "city"]

for c in cols_to_check:
    print(f"\n===== {c} =====")
    df.select(c).distinct().orderBy(c).display()


In [0]:
# remove empty spaces at beginning and end of string columns
log("Silver", "Cleaning string columns")

df = df.select([
    F.trim(col(c)).alias(c) if dict(df.dtypes)[c] == "string" else col(c)
    for c in df.columns
])

## Derived Columns

### Creating First Name and Last Name

In [0]:
if "customer_name" in df.columns:
    log("Silver", "Splitting customer_name")

    name_split = F.split(col("customer_name"), " ")

    df = (
        df
        .withColumn("customer_first_name", name_split[0])
        .withColumn("customer_last_name", name_split[F.size(name_split) - 1])
        .drop("customer_name")
    )

### Partition order_date


In [0]:
log("Silver", "Adding order_date partition columns")

df = (
    df
    .withColumn("year", F.year("order_date"))
    .withColumn("month", F.month("order_date"))
    .withColumn("day", F.dayofmonth("order_date"))
)

# Writing to Silver

In [0]:
log("Silver", f"Final rows before write: {df.count()}")
log("Silver", "Writing to silver.orders")

row_count = df.count()

if row_count == 0:
    raise ValueError("Silver dataset is empty - stopping pipeline")

try:
    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .partitionBy("year", "month", "day") \
        .saveAsTable("silver.orders")

    log("Silver", f"Write completed successfully (rows_written= {row_count})")

except Exception as e:
    log("Silver", f"ERROR during write: {str(e)}")
    raise